In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine

In [3]:
DB_USER = "postgres"
DB_PASSWORD = ""
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "onee_predictive_maintenance"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("PostgreSQL engine created successfully!")

PostgreSQL engine created successfully!


In [4]:
df = pd.read_sql(
    "SELECT * FROM iot_measurements;",
    engine
)

print("Dataset loaded successfully!")
print(df.shape)

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
potential_leakage_columns = [
    "failure_probability"
]

In [ ]:
df.drop(
    columns=potential_leakage_columns,
    inplace=True
)

Séparer Water et Electricity

In [ ]:
df_electricity = df[
    df["network_type"] == "Electricity"
].copy()

df_water = df[
    df["network_type"] == "Water"
].copy()

print("Electricity:", df_electricity.shape)
print("Water:", df_water.shape)

4. Supprimer les colonnes non applicables

electricité

In [ ]:
df_electricity.drop(
    columns=[
        "water_flow_l_min",
        "pressure_bar"
    ],
    inplace=True
)

eau

In [ ]:
df_water.drop(
    columns=[
        "voltage_v",
        "current_a",
        "frequency_hz",
        "power_factor",
        "energy_consumption_kwh"
    ],
    inplace=True
)

In [ ]:
print(
    df_electricity.isnull()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

print(
    df_water.isnull()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

Traiter les vraies valeurs manquantes

In [ ]:
numeric_cols_electricity = (
    df_electricity
    .select_dtypes(include=np.number)
    .columns
)

for col in numeric_cols_electricity:
    df_electricity[col] = (
        df_electricity[col]
        .fillna(df_electricity[col].median())
    )

In [ ]:
numeric_cols_water = (
    df_water
    .select_dtypes(include=np.number)
    .columns
)

for col in numeric_cols_water:
    df_water[col] = (
        df_water[col]
        .fillna(df_water[col].median())
    )

In [ ]:
df_electricity["vibration_mm_s"] = (
    df_electricity
    .groupby("equipment_type")["vibration_mm_s"]
    .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
df.groupby("target_failure")[
    [
        "vibration_mm_s",
        "voltage_v",
        "pressure_bar"
    ]
].describe()

In [ ]:
df[
    df["voltage_v"] > 1000
]["target_failure"].value_counts(normalize=True)

In [ ]:
df[
    df["vibration_mm_s"] > 20
]["target_failure"].value_counts(normalize=True)

Feature Engineering

In [ ]:
df_electricity["maintenance_overdue"] = (
    df_electricity["maintenance_delay_days"] > 30
).astype(int)

df_electricity["usage_intensity"] = (
    df_electricity["operating_hours"]
    / (df_electricity["equipment_age_years"] + 1)
)

df_electricity["vibration_age_interaction"] = (
    df_electricity["vibration_mm_s"]
    * df_electricity["equipment_age_years"]
)

print(
    df_electricity[
        [
            "maintenance_overdue",
            "usage_intensity",
            "vibration_age_interaction"
        ]
    ].head()
)

Encoder les variables catégorielles

In [ ]:
categorical_columns = [
    "city",
    "district",
    "equipment_type",
    "current_condition"
]
print("Categorical columns:")
print(categorical_columns)

suppression des identifiants inutiles 

In [ ]:
columns_to_drop = [
    "measurement_id",
    "sensor_id"
]
print("Columns to exclude from modeling:")
print(columns_to_drop)

## Data Leakage Analysis

Potential data leakage variables are analyzed before model training. 
The objective is to identify features that may directly or indirectly contain information used to generate the target variable.

In [ ]:
df_electricity[
    ["anomaly_score", "target_failure"]
].corr()

In [ ]:
df_water[
    ["anomaly_score", "target_failure"]
].corr()

In [ ]:
df_electricity.groupby("target_failure")["anomaly_score"].describe()

In [ ]:
df_water.groupby("target_failure")["anomaly_score"].describe()

## Outlier Analysis and Treatment

Extreme sensor measurements were identified during the exploratory data analysis. 
Outliers are analyzed before applying any transformation in order to preserve potentially meaningful failure-related observations.

electricity

In [ ]:
electricity_outlier_columns = [
    "voltage_v",
    "current_a",
    "vibration_mm_s",
    "noise_db"
]

df_electricity[electricity_outlier_columns].describe(
    percentiles=[0.01, 0.05, 0.95, 0.99]
).T

In [ ]:
for col in electricity_outlier_columns:
    q1 = df_electricity[col].quantile(0.25)
    q3 = df_electricity[col].quantile(0.75)
    
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = (
        (df_electricity[col] < lower_bound) |
        (df_electricity[col] > upper_bound)
    ).sum()
    
    print(
        col,
        "->",
        outliers,
        "outliers",
        f"({outliers / len(df_electricity) * 100:.2f}%)"
    )

water

In [ ]:
water_outlier_columns = [
    "pressure_bar",
    "water_flow_l_min",
    "vibration_mm_s",
    "noise_db"
]

df_water[water_outlier_columns].describe(
    percentiles=[0.01, 0.05, 0.95, 0.99]
).T

for col in water_outlier_columns:
    q1 = df_water[col].quantile(0.25)
    q3 = df_water[col].quantile(0.75)
    
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = (
        (df_water[col] < lower_bound) |
        (df_water[col] > upper_bound)
    ).sum()
    
    print(
        col,
        "->",
        outliers,
        "outliers",
        f"({outliers / len(df_water) * 100:.2f}%)"
    )

Vérifier si les outliers sont liés aux pannes

electricity


In [ ]:
for col in electricity_outlier_columns:
    threshold = df_electricity[col].quantile(0.99)

    normal_failure_rate = df_electricity[
        df_electricity[col] <= threshold
    ]["target_failure"].mean()

    extreme_failure_rate = df_electricity[
        df_electricity[col] > threshold
    ]["target_failure"].mean()

    print(f"\n{col}")
    print("Normal failure rate :", round(normal_failure_rate, 4))
    print("Extreme failure rate:", round(extreme_failure_rate, 4))

water

In [ ]:
for col in water_outlier_columns:
    threshold = df_water[col].quantile(0.99)

    normal_failure_rate = df_water[
        df_water[col] <= threshold
    ]["target_failure"].mean()

    extreme_failure_rate = df_water[
        df_water[col] > threshold
    ]["target_failure"].mean()

    print(f"\n{col}")
    print("Normal failure rate :", round(normal_failure_rate, 4))
    print("Extreme failure rate:", round(extreme_failure_rate, 4))

## Extreme Sensor Feature Engineering

The outlier analysis showed that extreme vibration and noise measurements are strongly associated with equipment failures, particularly in the water network.

Therefore, these extreme observations are preserved and transformed into additional binary risk indicators rather than being removed.

In [ ]:
for data in [df_electricity, df_water]:
    vibration_threshold = data["vibration_mm_s"].quantile(0.99)
    noise_threshold = data["noise_db"].quantile(0.99)

    data["extreme_vibration"] = (
        data["vibration_mm_s"] > vibration_threshold
    ).astype(int)

    data["extreme_noise"] = (
        data["noise_db"] > noise_threshold
    ).astype(int)

In [ ]:
print(
    df_electricity[
        ["extreme_vibration", "extreme_noise"]
    ].sum()
)

print(
    df_water[
        ["extreme_vibration", "extreme_noise"]
    ].sum()
)

In [ ]:
df_electricity = df_electricity.drop(
    columns=columns_to_drop,
    errors="ignore"
)

df_water = df_water.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Electricity shape:", df_electricity.shape)
print("Water shape:", df_water.shape)

In [ ]:
print("Electricity columns:")
print(df_electricity.columns.tolist())

print("\nWater columns:")
print(df_water.columns.tolist())

## Removal of Irrelevant Identifiers

The `measurement_id` and `sensor_id` columns were removed before the machine learning phase.

These variables represent unique identifiers and do not directly describe the operational condition of the equipment. Keeping them could introduce noise and lead the model to learn artificial patterns related to identifiers rather than meaningful failure indicators.

## Traitement de measurement_timestamp

## Temporal Feature Engineering

Temporal information is extracted from the measurement timestamp to capture potential seasonal and operational patterns associated with equipment failures.

In [ ]:
leakage_columns = [
    "failure_probability",
    "anomaly_score",
    "failure_history",
    "target_failure"
]

print("Electricity:")
print(
    df_electricity[leakage_columns]
    .corr()["target_failure"]
    .sort_values(ascending=False)
)

print("\nWater:")
print(
    df_water[leakage_columns]
    .corr()["target_failure"]
    .sort_values(ascending=False)
)

In [ ]:
print(df_electricity.columns.tolist())

In [ ]:
leakage_columns = [
    "anomaly_score",
    "target_failure"
]

print("Electricity:")
print(
    df_electricity[leakage_columns]
    .corr()["target_failure"]
    .sort_values(ascending=False)
)

print("\nWater:")
print(
    df_water[leakage_columns]
    .corr()["target_failure"]
    .sort_values(ascending=False)
)

In [ ]:
print("Electricity:")
print(
    df_electricity.groupby("target_failure")[
        ["anomaly_score"]
    ].mean()
)

print("\nWater:")
print(
    df_water.groupby("target_failure")[
        ["anomaly_score"]
    ].mean()
)

# 5. Data Leakage Analysis

Potential data leakage variables are investigated before the machine learning phase.

Variables such as `failure_probability` and `anomaly_score` may contain information strongly related to the target variable. Their relationship with `target_failure` is analyzed to determine whether they should be retained or removed from the predictive model.

In [ ]:
df_water["maintenance_overdue"] = (
    df_water["maintenance_delay_days"] > 30
).astype(int)

df_water["usage_intensity"] = (
    df_water["operating_hours"]
    / (df_water["equipment_age_years"] + 1)
)

df_water["vibration_age_interaction"] = (
    df_water["vibration_mm_s"]
    * df_water["equipment_age_years"]
)

print(
    df_water[
        [
            "maintenance_overdue",
            "usage_intensity",
            "vibration_age_interaction"
        ]
    ].head()
)

## Anomaly Score Monitoring

The `anomaly_score` variable is retained at this stage because its correlation with the target remains moderate.

However, a robustness experiment will be conducted during the modeling phase by comparing model performance with and without this feature to assess potential indirect data leakage.

In [ ]:
extreme_columns = [
    "extreme_vibration",
    "extreme_noise"
]

df_electricity = df_electricity.drop(
    columns=extreme_columns,
    errors="ignore"
)

df_water = df_water.drop(
    columns=extreme_columns,
    errors="ignore"
)

print(
    "Electricity extreme columns:",
    [col for col in extreme_columns if col in df_electricity.columns]
)

print(
    "Water extreme columns:",
    [col for col in extreme_columns if col in df_water.columns]
)

In [ ]:
print("df" in globals())
print("df_electricity" in globals())
print("df_water" in globals())

In [5]:
import os

os.makedirs("../data/processed", exist_ok=True)

df_electricity.to_csv(
    "../data/processed/electricity_preprocessed.csv",
    index=False
)

df_water.to_csv(
    "../data/processed/water_preprocessed.csv",
    index=False
)

print("Datasets saved successfully!")

NameError: name 'df_electricity' is not defined